# CMIE DOLPHIN

Run the current DOLPHIN algorithm on a CMIE household panel. This notebook is self-contained except for the `transition_subgroup_discovery` source folder and the CMIE CSV/table created on the server.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

# If the notebook is launched outside the copied project folder, set this manually, e.g.
# PROJECT_ROOT = Path("/home/user/my_project")
PROJECT_ROOT = None

def find_project_root(start):
    candidates = [Path(start).resolve(), *Path(start).resolve().parents]
    for candidate in candidates:
        if (candidate / "transition_subgroup_discovery" / "src" / "tsd").exists():
            return candidate
        if candidate.name == "transition_subgroup_discovery" and (candidate / "src" / "tsd").exists():
            return candidate.parent
    raise FileNotFoundError(
        "Could not find transition_subgroup_discovery/src/tsd. "
        "Set PROJECT_ROOT to the folder that contains transition_subgroup_discovery."
    )

REPO = Path(PROJECT_ROOT).resolve() if PROJECT_ROOT is not None else find_project_root(Path.cwd())
SRC = REPO / "transition_subgroup_discovery" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tsd.features import apply_target, build_temporal_features
from tsd.io import ensure_dir, load_json, save_json
from tsd.dolphin import run_dolphin, run_dolphin_binary

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 220)
print("Repo:", REPO)
print("TSD source:", SRC)

## Configure paths and columns

If you run this after the original CMIE extraction notebook, it will automatically use `panel_df` or `csv_path` if those variables exist. Otherwise set `RAW_CSV_PATH` manually.

In [ ]:
ID_COL = "HH_ID"
DATE_COL = "MONTH_SLOT_DATE"
TARGET_COL = "TOT_INC"
TARGET_NAME = "total_income"

# Change this on the server if you want to read directly from a saved CSV.
RAW_CSV_PATH = Path("cmie_panel.csv")

DATA_DIR = REPO / "transition_subgroup_discovery" / "data"
CONFIG_PATH = REPO / "transition_subgroup_discovery" / "configs" / "cmie.generated.json"
PREPARED_CSV_PATH = DATA_DIR / "cmie_dolphin_panel.csv"
OUTPUT_DIR = REPO / "transition_subgroup_discovery" / "outputs" / "cmie"

# These are converted to one-hot numeric covariates if present.
CATEGORICAL_COLUMNS = ["STATE", "REGION_TYPE", "MAX_EDU", "dominant_source", "income_quintile"]
MAX_EDU_ORDER = {
    "None or Primary": 1.0,
    "Class 10": 2.0,
    "Class 12 / Diploma": 3.0,
    "Graduate and Above": 4.0,
    "Unknown": np.nan,
}

# Keep this compact. Empty means all numeric columns, which makes CMIE rules too long.
INCLUDE_COLUMNS = [
    "wage_share",
    "biz_share",
    "transfer_share",
    "capital_income_share",
    "EMPLOYED_N",
    "TOT_N",
    "MAX_EDU_LEVEL",
]

print("Prepared CSV will be written to:", PREPARED_CSV_PATH)
print("Generated config will be written to:", CONFIG_PATH)
print("Outputs will be written to:", OUTPUT_DIR)

## Prepare CMIE panel

This cell keeps the algorithm target-independent: the target is only used as `TOT_INC`; explanatory rules come from independent covariates and their temporal summaries.

In [ ]:
def classify_dominant_source(df):
    required = [
        "TOT_INC",
        "INC_OF_ALL_MEMS_FRM_WAGES",
        "INC_OF_HH_FRM_BIZ_PROFIT",
        "INC_OF_HH_FRM_GOVT_TRF",
        "INC_OF_HH_FRM_PVT_TRF",
        "INC_OF_HH_FRM_NGO_TRF",
        "INC_OF_HH_FRM_IN_KIND_GOVT_TRF",
        "INC_OF_ALL_MEMS_FRM_DIVIDEND",
        "INC_OF_ALL_MEMS_FRM_INTEREST",
        "INC_OF_ALL_MEMS_FRM_FD_PF_INS",
        "INC_OF_HH_FRM_RENT",
    ]
    if not all(c in df.columns for c in required):
        return df
    out = df.copy()
    out["TOT_INC"] = pd.to_numeric(out["TOT_INC"], errors="coerce")
    safe_income = out["TOT_INC"].where(out["TOT_INC"] > 0, np.nan)
    wage_share = pd.to_numeric(out["INC_OF_ALL_MEMS_FRM_WAGES"], errors="coerce") / safe_income
    biz_share = pd.to_numeric(out["INC_OF_HH_FRM_BIZ_PROFIT"], errors="coerce") / safe_income
    transfer_total = sum(pd.to_numeric(out[c], errors="coerce").fillna(0) for c in ["INC_OF_HH_FRM_GOVT_TRF", "INC_OF_HH_FRM_PVT_TRF", "INC_OF_HH_FRM_NGO_TRF", "INC_OF_HH_FRM_IN_KIND_GOVT_TRF"])
    capital_total = sum(pd.to_numeric(out[c], errors="coerce").fillna(0) for c in ["INC_OF_ALL_MEMS_FRM_DIVIDEND", "INC_OF_ALL_MEMS_FRM_INTEREST", "INC_OF_ALL_MEMS_FRM_FD_PF_INS", "INC_OF_HH_FRM_RENT"])
    transfer_share = transfer_total / safe_income
    capital_share = capital_total / safe_income
    out["dominant_source"] = np.select(
        [wage_share >= 0.8, biz_share >= 0.5, transfer_share >= 0.5, capital_share >= 0.3],
        ["wage_dominant", "business_dominant", "transfer_dominant", "capital_dominant"],
        default="mixed",
    )
    return out

def repair_copy_pasted_csv_frame(df):
    if df.shape[1] != 1 or "," not in str(df.columns[0]):
        return df
    from io import StringIO
    lines = [str(df.columns[0])] + df.iloc[:, 0].astype(str).tolist()
    return pd.read_csv(StringIO("\n".join(lines)))

if "panel_df" in globals():
    raw = panel_df.copy()
    print("Using panel_df from the current notebook session.")
elif "csv_path" in globals() and Path(csv_path).exists():
    raw = repair_copy_pasted_csv_frame(pd.read_csv(csv_path))
    print("Loaded CSV from existing csv_path:", csv_path)
elif RAW_CSV_PATH.exists():
    raw = repair_copy_pasted_csv_frame(pd.read_csv(RAW_CSV_PATH))
    print("Loaded CSV from RAW_CSV_PATH:", RAW_CSV_PATH)
else:
    raise FileNotFoundError("No CMIE panel found. Define panel_df, csv_path, or set RAW_CSV_PATH to an existing CSV.")

work = raw.copy()
work[ID_COL] = pd.to_numeric(work[ID_COL], errors="coerce")
work[DATE_COL] = pd.to_datetime(work[DATE_COL], errors="coerce")
work[TARGET_COL] = pd.to_numeric(work[TARGET_COL], errors="coerce")
work = classify_dominant_source(work)

if "TOT_INC" in work.columns and "income_quintile" not in work.columns:
    baseline = work.sort_values(DATE_COL).groupby(ID_COL, as_index=False).tail(1)
    try:
        quintiles = pd.qcut(baseline[TARGET_COL], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
        work = work.merge(pd.DataFrame({ID_COL: baseline[ID_COL].values, "income_quintile": quintiles.astype(str).values}), on=ID_COL, how="left")
    except ValueError:
        pass

if "MAX_EDU" in work.columns:
    work["MAX_EDU_LEVEL"] = work["MAX_EDU"].astype("string").map(MAX_EDU_ORDER).astype(float)
dummy_cats = [c for c in ["dominant_source", "income_quintile"] if c in work.columns]
if dummy_cats:
    dummies = pd.get_dummies(work[dummy_cats].astype("string").fillna("Unknown"), prefix=dummy_cats, dummy_na=False, dtype=float)
    work = pd.concat([work.drop(columns=dummy_cats), dummies], axis=1)
drop_cats = [c for c in ["STATE", "REGION_TYPE", "MAX_EDU"] if c in work.columns]
if drop_cats:
    work = work.drop(columns=drop_cats)

work = work.dropna(subset=[ID_COL, DATE_COL, TARGET_COL]).copy()
numeric_cols = [c for c in work.columns if c in {ID_COL, DATE_COL} or pd.to_numeric(work[c], errors="coerce").notna().any()]
work = work[numeric_cols].copy()
for col in work.columns:
    if col not in {ID_COL, DATE_COL}:
        work[col] = pd.to_numeric(work[col], errors="coerce")

# Collapse accidental duplicate household-month rows by averaging numeric values.
value_cols = [c for c in work.columns if c not in {ID_COL, DATE_COL}]
work = work.groupby([ID_COL, DATE_COL], as_index=False)[value_cols].mean()

ensure_dir(DATA_DIR)
work.to_csv(PREPARED_CSV_PATH, index=False)

print("Rows:", len(work))
print("Entities:", work[ID_COL].nunique())
print("Date range:", work[DATE_COL].min(), "to", work[DATE_COL].max())
print("Numeric covariate count:", len([c for c in work.columns if c not in {ID_COL, DATE_COL, TARGET_COL}]))
display(work.head())

## Write CMIE config

In [ ]:
base_config = load_json(REPO / "transition_subgroup_discovery" / "configs" / "cmie.json")
base_config["data"]["path"] = str(PREPARED_CSV_PATH)
base_config["data"]["id_col"] = ID_COL
base_config["data"]["date_col"] = DATE_COL
base_config["output_dir"] = str(OUTPUT_DIR)
base_config["feature_engineering"]["time_unit"] = "month"
base_config["feature_engineering"]["include_columns"] = INCLUDE_COLUMNS
base_config["feature_engineering"]["include_prefixes"] = ["dominant_source_", "income_quintile_"]
base_config["feature_engineering"]["static_prefixes"] = ["dominant_source_", "income_quintile_"]
base_config["feature_engineering"]["lags"] = [6, 12, 18]
base_config["feature_engineering"]["windows"] = [6, 12, 18]
base_config["methods"]["dolphin"]["time_unit_label"] = "month"
base_config["methods"]["dolphin"]["pareto_filter"] = False
base_config["methods"]["dolphin"]["min_support"] = 0.01
base_config["methods"]["dolphin"]["max_support"] = 0.30
base_config["methods"]["dolphin"]["coverage_decay"] = 0.05
base_config["methods"]["dolphin"]["min_baseline_effect_size"] = 0.15
base_config["methods"]["dolphin"]["max_rule_jaccard"] = 0.8
base_config["methods"]["dolphin"]["min_rule_trajectory_distance"] = 0.08
base_config["targets"] = [{"name": TARGET_NAME, "mode": "column", "column": TARGET_COL}]

ensure_dir(CONFIG_PATH.parent)
save_json(CONFIG_PATH, base_config)
print(CONFIG_PATH.read_text(encoding="utf-8"))

## Run DOLPHIN

In [ ]:
cfg = load_json(CONFIG_PATH)
raw = pd.read_csv(cfg["data"]["path"])
target_cfg = cfg["targets"][0]
work_target, target_col, target_excludes = apply_target(raw, target_cfg)
table, feature_names = build_temporal_features(
    work_target,
    id_col=cfg["data"]["id_col"],
    date_col=cfg["data"]["date_col"],
    target_col=target_col,
    feature_cfg=cfg["feature_engineering"],
    exclude_cols=target_excludes,
)

print("Feature count:", len(feature_names))
metrics = run_dolphin(
    table=table,
    id_col=cfg["data"]["id_col"],
    target_col=target_col,
    feature_names=feature_names,
    cfg=cfg["methods"]["dolphin"],
    output_dir=Path(cfg["output_dir"]) / target_cfg["name"] / "dolphin",
)
display(pd.Series(metrics).to_frame("value"))

## Rules and diagnostics

In [ ]:
RUN_DIR = Path(cfg["output_dir"]) / TARGET_NAME / "dolphin"
rules = pd.read_csv(RUN_DIR / "rules.csv")
display(rules)
print((RUN_DIR / "rules_natural_language.txt").read_text(encoding="utf-8"))
print("Artifacts:", RUN_DIR)

## Plots

In [ ]:
plot_names = [
    "performance_vs_questions.png",
    "subgroup_distribution.png",
    "subgroup_distribution_overlay.png",
    "subgroup_evidence_summary.png",
]
for name in plot_names:
    path = RUN_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print("Missing:", name)

plots_dir = RUN_DIR / "plots"
if plots_dir.exists():
    for path in sorted(plots_dir.glob("*.png")):
        print(path.name)
        display(Image(filename=str(path)))

## Forest trees

In [ ]:
trees_dir = RUN_DIR / "forest_trees"
manifest = trees_dir / "manifest.csv"
if manifest.exists():
    display(pd.read_csv(manifest))
for path in sorted(trees_dir.glob("tree_*.png")):
    print(path.name)
    display(Image(filename=str(path)))